# Clase 5 — Memoria, contexto y estado

En la Clase 3 el agente aprendió a **decidir** con reglas y un LLM. En la Clase 4 aprendió a **actuar** mediante herramientas controladas por Python. En esta clase agregaremos una capacidad: **recordar lo necesario para continuar una conversación**.

## Agenda

| Bloque | Tiempo | Resultado |
|---|---:|---|
| Explicación | 40 min | Comprender tipos de memoria, opciones de almacenamiento y arquitectura básica |
| Actividad grupal | 40 min | Presentar una propuesta de consultoría para un caso de negocio |

## Objetivos

- Comprender por qué una solución necesita memoria.
- Diferenciar los principales tipos de memoria.
- Elegir entre una variable local y distintos tipos de bases de datos.
- Definir qué conviene recordar, durante cuánto tiempo y con qué controles.
- Comunicar una propuesta de arquitectura en términos de negocio.

---
# Parte 1 — Explicación (40 minutos)

## 1. Retomamos el agente de la Clase 4

En la clase anterior, una persona podía pedir:

> «Asigná el ticket T001 a la cola de pagos».

El LLM proponía usar **asignar_ticket**; Python verificaba que la herramienta estuviera permitida y ejecutaba la función.

~~~mermaid
flowchart LR
    P[Persona] --> M[Mensaje]
    M --> D[El agente propone<br/>una acción]
    D --> V[Python valida]
    V --> H[Herramienta registrada]
    H --> R[Resultado]
~~~

Ese recorrido funciona para un pedido completo. El problema aparece en el siguiente turno:

> «¿Qué pasó con ese ticket?»

La herramienta no sabe qué significa *ese ticket* y el modelo no conserva automáticamente la conversación anterior. La aplicación necesita recordar que, para esa persona, el ticket activo era **T001**.

## 2. Qué significa memoria en una solución con agentes 

La memoria es información que la aplicación guarda para poder usarla después. No vive mágicamente dentro del LLM.

~~~mermaid
flowchart LR
    T1[Turno 1<br/>«Mi ticket es T001»] --> G[La aplicación guarda<br/>ticket activo = T001]
    G --> T2[Turno 2<br/>«¿Cómo sigue mi ticket?»]
    T2 --> C[La aplicación recupera T001<br/>y completa el contexto]
~~~

Antes de guardar algo conviene responder cuatro preguntas sencillas:

1. ¿Para qué lo necesitaremos más adelante?
2. ¿A qué persona o proceso pertenece?
3. ¿Durante cuánto tiempo seguirá siendo útil?
4. ¿Qué daño puede causar si es incorrecto, queda viejo o llega a otra persona?

> Recordar más no siempre mejora la solución. Una buena memoria conserva lo necesario y puede corregirse o eliminarse.

## 3. Tipos de memoria 

No toda la información cumple la misma función. Separarla ayuda a decidir qué guardar y cuándo eliminarla.

| Tipo | Qué contiene | Ejemplo | Duración habitual | Riesgo principal |
|---|---|---|---|---|
| **Historial reciente** | últimos intercambios de la conversación | pregunta y respuesta anteriores | minutos u horas | acumular texto irrelevante o sensible |
| **Memoria de trabajo** | datos necesarios para la tarea actual | ticket activo T001, pedido P882 | hasta cerrar la tarea | usar un dato viejo en una acción nueva |
| **Estado del proceso** | paso en el que se encuentra el flujo | esperando confirmación | hasta completar o abandonar el proceso | ejecutar una acción fuera de orden |
| **Perfil** | preferencias estables y confirmadas | idioma o canal preferido | meses, con revisión | asumir que una preferencia sigue vigente |
| **Resumen** | hechos importantes de una conversación extensa | reclamo abierto por pedido P882 | según la política del negocio | resumir mal u omitir una condición |

### Una distinción importante

El estado de un pedido, el precio de un producto o una política de devolución no son recuerdos personales del agente. Son **datos del negocio** y deberían consultarse mediante una herramienta en su sistema de origen. La memoria puede guardar el identificador P882; la herramienta consulta su estado actualizado.

Esta separación mantiene la progresión:

- Clase 3: el agente interpreta y propone una decisión.
- Clase 4: una herramienta consulta o modifica el sistema del negocio.
- Clase 5: la memoria permite continuar el caso sin pedir todos los datos nuevamente.

## 4. Dónde se puede guardar la memoria

| Opción | Cuándo sirve | Ventaja | Limitación |
|---|---|---|---|
| **Variable local / DataFrame** | demostraciones y prototipos de una sola ejecución | simple y visible | se pierde al reiniciar y no sirve para varios servidores |
| **Archivo JSON o CSV** | prueba pequeña con pocos datos | fácil de abrir y compartir | se vuelve frágil con varios usuarios escribiendo al mismo tiempo |
| **Base relacional SQL** | usuarios, tickets, estados y relaciones claras | reglas, consultas y auditoría confiables | requiere diseñar tablas y mantener la base |
| **Base documental o clave-valor** | sesiones rápidas con estructuras que pueden cambiar | flexible y rápida; puede aplicar vencimiento automático | las relaciones y reportes complejos cuestan más |
| **Base vectorial** | buscar textos parecidos por significado | recupera conversaciones o documentos relacionados | no debe ser la fuente oficial de un estado o una transacción |

### Cómo elegir

- Si queremos enseñar la idea, una variable local alcanza.
- Si la información debe sobrevivir reinicios, necesitamos almacenamiento persistente.
- Si hay relaciones importantes —persona, ticket, pedido y auditoría— una base SQL suele ser clara.
- Si predominan sesiones breves con vencimiento, una base clave-valor puede ser conveniente.
- Si necesitamos encontrar texto relacionado, una base vectorial puede complementar a las anteriores, no reemplazarlas.

En una solución real pueden convivir varias opciones. La elección depende del volumen, la duración, las consultas necesarias, la posibilidad de borrar datos y el costo de una equivocación.

## 5. Arquitectura básica de la solución

~~~mermaid
flowchart LR
    U[Persona identificada] --> A[Aplicación]
    A --> M[(Memoria de la sesión)]
    M --> A
    A --> D[Reglas + LLM]
    D --> V[Validaciones]
    V --> T[Herramienta permitida]
    T --> B[(Sistema del negocio)]
    B --> A
    A --> U
~~~

La aplicación coordina el recorrido:

- identifica a la persona;
- recupera solamente la memoria necesaria;
- permite que el agente proponga una decisión;
- valida antes de ejecutar una herramienta;
- guarda únicamente los cambios confirmados;
- responde y deja preparado el próximo turno.

El LLM ayuda a interpretar lenguaje. La identidad, los permisos, las herramientas y los datos oficiales permanecen bajo control de la aplicación.

## 6. Política mínima de memoria

Toda propuesta debería aclarar:

- **Propósito:** para qué se guarda cada dato.
- **Alcance:** a qué usuario, conversación o proceso pertenece.
- **Duración:** cuándo vence o deja de ser útil.
- **Confirmación:** qué datos deben ser confirmados antes de usarse.
- **Protección:** qué información no debe guardarse.
- **Corrección y olvido:** cómo actualizar o eliminar lo almacenado.
- **Pruebas:** cómo comprobaremos que dos usuarios nunca comparten memoria.

Estas decisiones son más importantes que la tecnología elegida.

---
## Ejemplo mínimo — Memoria local visible

Este ejemplo no representa una base de datos de producción. Solo permite ver la idea: una variable local conserva un registro por usuario y un DataFrame facilita consultarlo.

In [ ]:
import pandas as pd

MEMORIA_LOCAL = [
    {"usuario_id": "U01", "ticket_activo": "T001",
     "cola": "pagos", "estado": "asignado"},
    {"usuario_id": "U02", "ticket_activo": None,
     "cola": None, "estado": "esperando_ticket"},
]

memoria_df = pd.DataFrame(MEMORIA_LOCAL)
display(memoria_df)

# Consultamos solamente la memoria de U01
display(memoria_df[memoria_df["usuario_id"] == "U01"])

---
# Parte 2 — Actividad grupal (30/40 minutos)

## Caso de consultoría: agente de atención para Mercado Norte

Mercado Norte vende por web y aplicación móvil. Su equipo recibe consultas sobre pagos, entregas, seguridad y cambios de productos. En las clases anteriores la empresa construyó un agente que puede clasificar mensajes y usar herramientas para consultar pedidos o abrir reclamos.

### Condiciones del negocio

- La empresa atiende por web y aplicación móvil.
- Solamente una persona identificada puede consultar un pedido.
- El agente puede usar **consultar_pedido**, **abrir_reclamo** y **consultar_reclamo**.
- Contraseñas y datos completos de tarjetas no deben llegar al LLM ni guardarse.
- El estado oficial del pedido siempre se consulta en el sistema de ventas.
- Dos clientes nunca deben compartir recuerdos.
- La empresa necesita corregir o eliminar la memoria cuando corresponda.

### Rol del grupo

Son el equipo consultor. Deben recomendar una arquitectura comprensible para responsables de negocio y tecnología. **No deben programarla.**

### La entrega
Propuesta de arquitectura en un diagrama en Mermaid: 
- Ver: https://mermaid.live
